# 04 · Webcam eye tracking: net gaze convergence and H1a–H1c

**What this notebook does.** Analyses the online webcam eye-tracking study exactly as preregistered on OSF (https://osf.io/hr6x5) and described in 3-9:
reading the session records and applying the exclusion rules; per-second features of the 12 clips as shown; data quality and preprocessing
(drift correction, video coordinates, central bias); **net gaze convergence** = NSS of a viewer's gaze on the leave-one-out consensus map of the
other viewers minus the person-based shuffled (chance-level) baseline; H1a (within-person high minus low K-pop excerpts) with its adequacy
conditions, sensitivity versions and leave-one-out; H1b (bin-level mixed model with perceptual controls); H1c (slope on the excerpt index over all 12
excerpts, including the 12-point permutation test) and D1; AOI (face / below-face / other); heatmaps; exploratory analyses; and additional checks
(time-shifted baseline, spatial concentration of the consensus, timing, familiarity, trial order).

**Run order.** Notebook 01 must have produced `web_clips/clips/C1–C12.mp4`, `web_clips/clip_key.xlsx` (private) and `tables/eyetracking_selection_12.xlsx`.
Put the exported session file in `OUT/gaze/kpop_eyetracking_all.json` (and hand-sent sessions in `OUT/gaze/manual/`). Run top to bottom.
Steps marked *optional / heavy* (participant saturation, heatmap videos) can be skipped without affecting later steps.
`USE_PILOT` and `TEST_CLONES` must stay `False` / `0` for the real analysis.

**Thesis.** 3-9-3 to 3-9-7; results in chapter 5 (Tables 5-2 to 5-16).

---

## ۰۴ · ردیابی نگاه با وبکم: همگرایی خالص نگاه و H1a تا H1c
تحلیل داده‌ی مطالعه‌ی برخط دقیقاً طبق پیش‌ثبت OSF و بخش ۳-۹: قواعد حذف، کیفیت داده و پیش‌پردازش، همگرایی خالص نگاه (NSS روی نقشه‌ی اجماع دیگران منهای خط پایه‌ی شخصی شانسی)، H1a با شرایط کفایت و نسخه‌های حساسیت، H1b، H1c (با آزمون جایگشت ۱۲ قطعه) و D1، ناحیه‌های مورد علاقه، نقشه‌های حرارتی، تحلیل‌های اکتشافی و وارسی‌های تکمیلی (خط پایه‌ی جابه‌جاشده در زمان، تمرکز فضایی اجماع). کلیدهای آزمایشی باید خاموش باشند.

### 1. Install libraries

**فارسی:** نصب کتابخانه‌ها.

*Thesis: appendix C*

In [ ]:
# 1. Install libraries
# Thesis: appendix C
%pip install -q numpy pandas scipy statsmodels matplotlib opencv-python scenedetect librosa imageio-ffmpeg openpyxl xlsxwriter

### 2. Imports

**فارسی:** بارگذاری کتابخانه‌ها.

*Thesis: —*

In [ ]:
# 2. Imports
# Thesis: —
import json, math, re, subprocess, sys, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2, matplotlib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import imageio_ffmpeg
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 40)
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

### 3. PATHS (the only cell to edit)

**فارسی:** مسیرها: تنها خانه‌ای که باید ویرایش شود.

*Thesis: 3-9, 3-10*

In [ ]:
# 3. PATHS
# Thesis: 3-9, 3-10
# Inputs (not distributed): gaze/kpop_eyetracking_all.json (all session records), gaze/manual/*.json, web_clips/clips/C1-C12.mp4,
# web_clips/clip_key.xlsx (private), tables/eyetracking_selection_12.xlsx, and for the exploratory window analysis per_video/ + metadata.
BASE  = Path.home() / 'KPOP'                       # same as the product notebook
OUT   = BASE / 'analysis_v2_0'
GZ    = OUT / 'gaze'; FIG = GZ / 'figures'; VID = GZ / 'heatmap_videos'
CLIPS_DIR = OUT / 'web_clips' / 'clips'
DATA_JSON = GZ / 'kpop_eyetracking_all.json'
SEL12 = OUT / 'tables' / 'eyetracking_selection_12.xlsx'
CLIP_KEY = OUT / 'web_clips' / 'clip_key.xlsx'          # PRIVATE key (blind name -> level); never publish
METADATA = BASE / 'MV_metadata_105_verified_v2.xlsx'   # only for the exploratory window analysis
MANUAL_DIR = GZ / 'manual'                          # sessions sent by hand (downloaded kpop_et_*.json)
for d in [GZ, FIG, VID]: d.mkdir(parents=True, exist_ok=True)

### 4. Settings fixed in the preregistration

**فارسی:** تنظیمات ثابت پیش‌ثبت (تغییر نکنند) و کلیدهای آزمایشی.

*Thesis: 3-9-5; appendix C (Table C-2)*

In [ ]:
# 4. Settings fixed in the preregistration (do not change) and test switches
# Thesis: 3-9-5; appendix C (Table C-2)
# --- settings fixed in the preregistration (do not change) ---
SIGMA_DEG   = 4.0      # Gaussian smoothing of individual maps (deg); sensitivity 3.0 for participants with error <= 3 deg
ACC_MAX_DEG = 4.0      # participant exclusion: post-calibration check error above this
TRIAL_VALID = 0.70     # trial exclusion: fewer valid samples than this share
MIN_SAMPLES = 3        # bins with fewer valid samples are dropped for that participant
N_SHUFFLE   = 500      # draws for the person-based shuffled baseline
POST_CUT    = 0.5      # seconds after each cut dropped in H1b
N_BOOT      = 2000
GRID_W      = 64       # map resolution (cells across the video width)
EXC_S       = 20       # excerpt length (s)
# --- test switches (keep False / 0 for the real analysis) ---
USE_PILOT   = False
TEST_CLONES = 0
RNG = np.random.default_rng(2026)
print('data file found:', DATA_JSON.exists(), '| clips:', len(list(CLIPS_DIR.glob('C*.mp4'))), '| selection file:', SEL12.exists())

### 5. Read data, exclusion rules, data flow, clip key

**فارسی:** خواندن داده، قواعد حذف شرکت‌کننده و آزمایه، جریان داده، کلید ۱۲ قطعه.

*Thesis: 3-9-3, 3-9-4; Tables 5-2, 5-3*

In [ ]:
# 5. Read the data, apply the exclusion rules (OSF 5.6), data flow, clip key on one index scale
# Thesis: 3-9-3, 3-9-4; Tables 5-2, 5-3
raw = json.loads(DATA_JSON.read_text(encoding='utf-8'))
recs = []
for r in (raw if isinstance(raw, list) else [raw]):
    d = r.get('data', r) if isinstance(r, dict) else r
    if isinstance(d, str): d = json.loads(d)
    if isinstance(d, dict) and d.get('participant_id'): recs.append(d)
print('sessions in file:', len(recs))
# sessions whose upload failed and were sent by hand (downloaded kpop_et_*.json) go in gaze/manual
seen = {d['participant_id'] for d in recs}
for p in sorted(MANUAL_DIR.glob('*.json')) if MANUAL_DIR.exists() else []:
    d = json.loads(p.read_text(encoding='utf-8'))
    if d.get('participant_id') and d['participant_id'] not in seen: recs.append(d); seen.add(d['participant_id']); print('manual file added:', p.name)
print('sessions total:', len(recs))

if TEST_CLONES:
    print(f'*** TEST MODE: every pilot session is cloned {TEST_CLONES} times with noise; numbers are meaningless ***')
    clones = []
    for d in recs:
        if not d.get('pilot'): continue
        for k in range(TEST_CLONES):
            c = json.loads(json.dumps(d)); c['participant_id'] = f"{d['participant_id']}_clone{k + 1}"
            for T in c.get('trials', []):
                for g in T['gaze']:
                    if g.get('x') is not None:
                        g['x'] += float(RNG.normal(0, 40)); g['y'] += float(RNG.normal(0, 40))
            clones.append(c)
    recs += clones

def ppd_of(g):   # CSS pixels per degree; nominal fallback if the virtual chinrest was implausible
    return g.get('px_per_deg') if g.get('card_plausible') and g.get('dist_plausible') and g.get('px_per_deg') else g.get('px_per_deg_fallback') or g.get('px_per_deg')

def vc_err(v, key, f, ax=None):   # validation error summary; recomputed from the raw validation points if the field is missing
    if v.get(key) is not None: return v[key]
    pt = v.get('points') or []
    if not pt: return np.nan
    return float(f([abs(o['g' + ax] - o['t' + ax]) if ax else o['err'] for o in pt]))

P, TR, S = [], [], []
for d in recs:
    pid = d['participant_id']; g = d.get('geometry') or {}; ppd = ppd_of(g)
    vals = d.get('validations', [])
    checks = [v for v in vals if str(v.get('label', '')).startswith(('cal_check', 'post_calibration'))]
    passed = [v for v in checks if v.get('passed', True)]
    vc = passed[-1] if passed else (checks[-1] if checks else None)
    acc_px = vc.get('mean_err_px') if vc else np.nan; prec_px = vc.get('precision_px') if vc else np.nan
    end = next((v for v in vals if v.get('label') == 'end_session'), None); mid = next((v for v in vals if v.get('label') == 'mid_session'), None)
    info = d.get('info') or {}
    P.append(dict(pid=pid, pilot=bool(d.get('pilot')), version=d.get('version'), started=d.get('started'), screened_out=d.get('screened_out'),
                  age=info.get('age'), gender=info.get('gender'), vision=info.get('vision'), kpop_freq=info.get('kpop_freq'), fan=info.get('fan'),
                  ppd=ppd, geometry_nominal=not (g.get('card_plausible') and g.get('dist_plausible')), no_card=bool(g.get('no_card')),
                  view_dist_mm=g.get('view_dist_mm'), acc_px=acc_px, prec_px=prec_px,
                  acc_deg=acc_px / ppd if ppd else np.nan, prec_deg=prec_px / ppd if ppd else np.nan,
                  mid_err_deg=(mid['mean_err_px'] / ppd) if mid and ppd else np.nan, end_err_deg=(end['mean_err_px'] / ppd) if end and ppd else np.nan,
                  max_err_deg=vc_err(vc, 'max_err_px', np.max) / ppd if vc and ppd else np.nan,
                  err_x_deg=vc_err(vc, 'mean_err_x_px', np.mean, 'x') / ppd if vc and ppd else np.nan, err_y_deg=vc_err(vc, 'mean_err_y_px', np.mean, 'y') / ppd if vc and ppd else np.nan,
                  n_calibrations=len(d.get('calibration', [])), preload_s=(d.get('env') or {}).get('preload_ms', np.nan) / 1000,
                  latin_row=d.get('latin_row'), open_text=(d.get('open') or {}).get('text', ''), n_trials=len(d.get('trials', []))))
    for T in d.get('trials', []):
        rt = T.get('rating') or {}; dr = T.get('drift') or {}; R = T.get('rect') or {}
        fr = T.get('frames') or {}
        TR.append(dict(pid=pid, clip=T['clip'], order=T['order'], valid_share=T.get('valid_share', 0), rate_hz=T.get('rate_hz'),
                       left_fullscreen=bool(T.get('left_fullscreen')), tab_hidden=bool(T.get('tab_hidden')), stalls=T.get('stalls', 0) or 0,
                       muted=bool(T.get('muted_fallback')), frames_total=fr.get('total'), frames_dropped=fr.get('dropped'),
                       drift_dx=dr.get('dx', 0) or 0, drift_dy=dr.get('dy', 0) or 0, rect_l=R.get('left'), rect_t=R.get('top'), rect_w=R.get('width'), rect_h=R.get('height'),
                       vw=(T.get('viewport') or {}).get('w'), vh=(T.get('viewport') or {}).get('h'),
                       fam_song=rt.get('fam_song'), fam_group=rt.get('fam_group'), liking=rt.get('liking'),
                       sentence=T.get('sentence_end') or rt.get('sentence') or '', frame_log=T.get('frame_log', []), cross=T.get('cross', [])))
        for s in T['gaze']:
            if s.get('x') is not None: S.append((pid, T['clip'], s['tw'], s['vt'], s['x'], s['y']))
P = pd.DataFrame(P); TR = pd.DataFrame(TR)
S = pd.DataFrame(S, columns=['pid', 'clip', 'tw', 'vt', 'x', 'y'])

# --- exclusion rules (OSF 5.6) ---
P['excl'] = ''
P.loc[P.pilot & ~USE_PILOT, 'excl'] = 'pilot'
P.loc[(P.excl == '') & P.screened_out.notna(), 'excl'] = 'screened out: ' + P.screened_out.astype(str)
P.loc[(P.excl == '') & (P.n_trials == 0), 'excl'] = 'no trials'
P.loc[(P.excl == '') & ~(P.acc_deg <= ACC_MAX_DEG), 'excl'] = f'calibration error > {ACC_MAX_DEG} deg'
TR['trial_ok'] = (TR.valid_share >= TRIAL_VALID) & ~TR.left_fullscreen & ~TR.tab_hidden & (TR.stalls == 0)
TR['trial_excl'] = np.select([TR.valid_share < TRIAL_VALID, TR.left_fullscreen, TR.tab_hidden, TR.stalls > 0],
                             ['valid samples < 70%', 'left fullscreen', 'tab hidden', 'playback stall'], '')
bad_half = TR.groupby('pid').trial_ok.mean() < 0.5
P.loc[(P.excl == '') & P.pid.map(bad_half).fillna(False), 'excl'] = 'more than half of the trials excluded'
P['included'] = P.excl == ''
INC = list(P.loc[P.included, 'pid'])
TRI = TR[TR.pid.isin(INC) & TR.trial_ok].copy()

flow = pd.concat([pd.DataFrame([dict(step='sessions recorded', n=len(P))]),
                  P[P.excl != ''].groupby('excl').size().rename('n').reset_index().rename(columns={'excl': 'step'}).assign(step=lambda x: 'excluded: ' + x.step),
                  pd.DataFrame([dict(step='participants analysed', n=len(INC)), dict(step='trials analysed', n=len(TRI)),
                                dict(step='trials excluded (of included participants)', n=int((~TR[TR.pid.isin(INC)].trial_ok).sum()))])])
display(flow)
# site version per session and the reason recorded for each failed calibration attempt (site 1.8.4 onwards)
FAILS = pd.DataFrame([dict(pid=d['participant_id'], version=d.get('version'), attempt=c.get('pass'), reason=(c.get('fail') or {}).get('code'), lum=(c.get('fail') or {}).get('lum'))
                      for d in recs for c in d.get('calibration', []) if c.get('fail')])
display(P.groupby('version', dropna=False).agg(sessions=('pid', 'size'), included=('included', 'sum'), screened_out=('screened_out', lambda x: x.notna().sum())))
if len(FAILS): display(FAILS.groupby(['version', 'reason'], dropna=False).size().rename('failed attempts').reset_index())

# --- clip key: index on one scale for all 12, level, gender, corpus ---
SEL = pd.read_excel(SEL12); KEY = pd.read_excel(CLIP_KEY)
CK = SEL.rename(columns={'clip_id': 'clip'})[['clip', 'file_name', 'corpus', 'level', 'gender', 'excerpt_index_z', 'chorus_start']].copy()
CK['gender'] = CK.gender.str.lower(); CK['female'] = (CK.gender == 'female').astype(int)
CK['high'] = CK.level.isin(['high', 'indie_high']).astype(int)
CK['kpop'] = (CK.corpus == 'kpop').astype(int)
C9 = CK.loc[CK.file_name.str.contains('Chang Kiha|장기하'), 'clip'].tolist()
CLIPS = sorted(CK['clip'], key=lambda c: int(c[1:]))
print('clips:', CLIPS, '| dark single-dancer clip (registered sensitivity):', C9)
display(CK[['clip', 'corpus', 'level', 'gender', 'excerpt_index_z']].sort_values('excerpt_index_z', ascending=False))

### 6. Per-second features of the 12 clips as shown

**فارسی:** ویژگی‌های هر ثانیه‌ی ۱۲ کلیپ همان‌طور که نمایش داده شدند: روشنایی، کنتراست، حرکت، برش، قدرت ضربه، چهره‌ها.

*Thesis: 3-9-6 (H1b controls), 5-10*

In [ ]:
# 6. Per-second features of the 12 clips exactly as shown (cached in gaze/clip_features.xlsx, gaze/faces/*.npz)
# Thesis: 3-9-6 (H1b controls), 5-10
from scenedetect import detect, ContentDetector
import librosa
FEAT_X = GZ / 'clip_features.xlsx'; FACE_D = GZ / 'faces'; FACE_D.mkdir(exist_ok=True)
YUNET = OUT / 'face_detection_yunet_2023mar.onnx'
if not YUNET.exists():
    try: urllib.request.urlretrieve('https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', YUNET)
    except Exception as e: print('YuNet download failed:', e)
FACE_FPS = 5.0

def clip_features(clip):
    path = CLIPS_DIR / f'{clip}.mp4'
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25; W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    try: det = cv2.FaceDetectorYN.create(str(YUNET), '', (320, 320), 0.6)
    except Exception as e: det = None; print('face detector unavailable, AOI face/body will be empty:', str(e)[:80])
    rows, faces, ftimes, prev, k, fstep = [], [], [], None, 0, max(1, round(fps / FACE_FPS))
    while True:
        ok, fr = cap.read()
        if not ok: break
        t = k / fps; sm = cv2.resize(fr, (320, int(320 * H / W))); gray = cv2.cvtColor(sm, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
        rows.append((t, gray.mean(), gray.std(), np.abs(gray - prev).mean() if prev is not None else np.nan)); prev = gray
        if det is not None and k % fstep == 0:
            det.setInputSize((fr.shape[1], fr.shape[0])); _, fc = det.detect(fr); ftimes.append(t)
            for f in ([] if fc is None else fc): faces.append((t, f[0] / W, f[1] / H, f[2] / W, f[3] / H, f[-1]))
        k += 1
    cap.release()
    fr_ = pd.DataFrame(rows, columns=['t', 'lum', 'con', 'mot'])
    cuts = [s[0].get_seconds() for s in detect(str(path), ContentDetector())][1:]
    wav = FACE_D / f'{clip}.wav'   # mp4 audio is read through ffmpeg (librosa cannot open mp4 on every system)
    subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-i', str(path), '-vn', '-ac', '1', '-ar', '22050', str(wav)], check=True)
    y, sr = librosa.load(str(wav), sr=22050, mono=True); wav.unlink()
    on = librosa.onset.onset_strength(y=y, sr=sr); ton = librosa.times_like(on, sr=sr)
    _, beats = librosa.beat.beat_track(onset_envelope=on, sr=sr); tbeat = librosa.frames_to_time(beats, sr=sr)
    b = np.floor(fr_.t).astype(int); fr_['bin'] = b
    per = fr_.groupby('bin').agg(lum=('lum', 'mean'), con=('con', 'mean'), mot=('mot', 'mean')).reset_index()
    per['cuts'] = [sum(1 for c in cuts if bb <= c < bb + 1) for bb in per.bin]
    per['onset'] = [on[(ton >= bb) & (ton < bb + 1)].mean() for bb in per.bin]
    np.savez(FACE_D / f'{clip}.npz', faces=np.array(faces, np.float32).reshape(-1, 6), ftimes=np.array(ftimes), cuts=np.array(cuts), beats=tbeat, fps=fps, W=W, H=H)
    return per.assign(clip=clip)

if FEAT_X.exists() and all((FACE_D / f'{c}.npz').exists() for c in CLIPS):
    FEAT = pd.read_excel(FEAT_X)
else:
    FEAT = pd.concat([clip_features(c) for c in CLIPS], ignore_index=True); FEAT.to_excel(FEAT_X, index=False)
FACES = {c: dict(np.load(FACE_D / f'{c}.npz')) for c in CLIPS}
CUTS = {c: FACES[c]['cuts'] for c in CLIPS}; BEATS = {c: FACES[c]['beats'] for c in CLIPS}
ASPECT = {c: float(FACES[c]['W'] / FACES[c]['H']) for c in CLIPS}
display(FEAT.groupby('clip')[['lum', 'con', 'mot', 'cuts', 'onset']].agg({'lum': 'mean', 'con': 'mean', 'mot': 'mean', 'cuts': 'sum', 'onset': 'mean'}).round(3).loc[CLIPS])

### 7. Data quality and preprocessing

**فارسی:** کیفیت داده: نرخ نمونه، هم‌ترازی فریم، صحت و دقت، جابه‌جایی، تمایل به مرکز؛ تصحیح جابه‌جایی و مختصات قاب.

*Thesis: 3-9-4; Table 5-4*

In [ ]:
# 7. Data quality (sampling rate, frame alignment, accuracy/precision, drift, central bias) and drift-corrected gaze in video coordinates
# Thesis: 3-9-4; Table 5-4
def frame_lag(row):   # gaze sample video time minus media time of the frame on screen at that moment (ms)
    fl = row.frame_log
    if not fl: return np.nan
    ftw = np.array([f['tw'] for f in fl]); fmt = np.array([f['mt'] for f in fl])
    s = S[(S.pid == row.pid) & (S['clip'] == row['clip'])]
    i = np.searchsorted(ftw, s.tw.values, side='right') - 1; ok = i >= 0
    return float(np.median(np.abs(s.vt.values[ok] - fmt[i[ok]])) * 1000) if ok.any() else np.nan
TR['frame_lag_ms'] = TR.apply(frame_lag, axis=1)
Q = TR[TR.pid.isin(P.pid)].groupby('pid').agg(rate_hz=('rate_hz', 'mean'), valid_share=('valid_share', 'mean'), frame_lag_ms=('frame_lag_ms', 'median'),
                                               frames_dropped=('frames_dropped', 'sum'), stalls=('stalls', 'sum')).reset_index()
Q['drift_px'] = TR.assign(dm=np.hypot(TR.drift_dx, TR.drift_dy)).groupby('pid').dm.median().reindex(Q.pid).values
Q = Q.merge(P[['pid', 'included', 'excl', 'acc_px', 'acc_deg', 'prec_px', 'prec_deg', 'max_err_deg', 'err_x_deg', 'err_y_deg', 'mid_err_deg', 'end_err_deg', 'geometry_nominal', 'no_card', 'view_dist_mm', 'n_calibrations', 'preload_s']], on='pid')
Q['drift_deg'] = Q.drift_px / P.set_index('pid').ppd.reindex(Q.pid).values
display(Q.round(2))
print('included participants: mean accuracy %.2f deg, precision %.2f deg, sampling %.1f Hz, frame lag %.0f ms' %
      (Q[Q.included].acc_deg.mean(), Q[Q.included].prec_deg.mean(), Q[Q.included].rate_hz.mean(), Q[Q.included].frame_lag_ms.median()))
print('included participants: max validation error %.2f deg, mean horizontal %.2f, vertical %.2f deg | nominal-geometry (ppd fallback): %d of %d' %
      (Q[Q.included].max_err_deg.mean(), Q[Q.included].err_x_deg.mean(), Q[Q.included].err_y_deg.mean(), Q[Q.included].geometry_nominal.sum(), Q.included.sum()))

# drift-corrected gaze in video coordinates (0-1); the first second, bins and post-cut flags
G = S[S.pid.isin(INC)].merge(TRI[['pid', 'clip', 'drift_dx', 'drift_dy', 'rect_l', 'rect_t', 'rect_w', 'rect_h', 'vw', 'vh']], on=['pid', 'clip'])
G['xc'] = G.x - G.drift_dx; G['yc'] = G.y - G.drift_dy
G['u'] = (G.xc - G.rect_l) / G.rect_w; G['v'] = (G.yc - G.rect_t) / G.rect_h
G['in_video'] = G.u.between(0, 1, inclusive='left') & G.v.between(0, 1, inclusive='left')
G = G[(G.vt >= 0) & (G.vt < EXC_S)].copy(); G['bin'] = np.floor(G.vt).astype(int)
G['post_cut'] = [any(0 <= vt - c < POST_CUT for c in CUTS[cl]) for vt, cl in zip(G.vt, G['clip'])]
G = G.merge(P[['pid', 'ppd']], on='pid')
G['dist_centre_deg'] = np.hypot(G.xc - G.vw / 2, G.yc - G.vh / 2) / G.ppd
CB = G.groupby('clip').agg(central_bias_deg=('dist_centre_deg', 'mean'), share_outside_video=('in_video', lambda x: 1 - x.mean())).reindex(CLIPS).reset_index()
display(CB.merge(CK[['clip', 'corpus', 'level']], on='clip').round(3))

### 8. Net gaze convergence (NSS minus person-based chance level)

**فارسی:** همگرایی خالص نگاه: NSS روی نقشه‌ی اجماع دیگران (بدون خود فرد) منهای خط پایه‌ی شخصی درهم‌ریخته (سطح شانس).

*Thesis: 3-9-5; Table 5-5*

In [ ]:
# 8. Net gaze convergence: NSS on the leave-one-out consensus map minus the person-based shuffled baseline
# Thesis: 3-9-5; Table 5-5
def grid_h(c): return max(8, int(round(GRID_W / ASPECT[c])))
RW = TRI.set_index(['pid', 'clip']).rect_w.to_dict(); PPD = P.set_index('pid').ppd.to_dict()

# build() takes the clip and in_video masks from the gaze table it is given (g_source), so sensitivity versions
# with another gaze table (drift-uncorrected, fixation-only, time-shifted) are masked correctly.
def build(pids, clips=None, sigma_deg=SIGMA_DEG, sigma_frac=None, bins=range(1, EXC_S), drop_post_cut=False, n_shuffle=N_SHUFFLE,
          seed=1, min_samples=MIN_SAMPLES, g_source=None, sigma_by_pid=None):
    clips = clips or CLIPS; G_ = G if g_source is None else g_source
    g = G_[G_.pid.isin(pids) & G_['clip'].isin(clips) & G_.in_video & G_.bin.isin(list(bins))]
    if drop_post_cut: g = g[~g.post_cut]
    cells = {}
    for (p, c, b), x in g.groupby(['pid', 'clip', 'bin']):
        if len(x) >= min_samples: cells[(p, c, b)] = (x.u.values, x.v.values)
    maps = {}
    for (p, c, b), (u, v) in cells.items():
        gh = grid_h(c); m = np.zeros((gh, GRID_W))
        np.add.at(m, (np.minimum((v * gh).astype(int), gh - 1), np.minimum((u * GRID_W).astype(int), GRID_W - 1)), 1)
        sd_deg = sigma_by_pid.get(p, sigma_deg) if sigma_by_pid else sigma_deg
        s = sigma_frac * GRID_W if sigma_frac else sd_deg * PPD[p] / RW[(p, c)] * GRID_W
        m = gaussian_filter(m, s, mode='constant'); maps[(p, c, b)] = m / m.sum()
    rng = np.random.default_rng(seed); out = []; by_cb, by_p = {}, {}
    for k in cells: by_p.setdefault(k[0], []).append(k)
    for (p, c, b) in maps: by_cb.setdefault((c, b), []).append(p)
    for (c, b), ps in by_cb.items():
        if len(ps) < 3: continue
        gh = grid_h(c); tot = sum(maps[(q, c, b)] for q in ps)
        for p in ps:
            C = (tot - maps[(p, c, b)]) / (len(ps) - 1); sd = C.std()
            if sd == 0: continue
            Z = (C - C.mean()) / sd
            def nss(u, v): return Z[np.minimum((v * gh).astype(int), gh - 1), np.minimum((u * GRID_W).astype(int), GRID_W - 1)].mean()
            obs = nss(*cells[(p, c, b)]); pool = [k for k in by_p[p] if k[1] != c]
            if not pool: continue
            pm = np.array([nss(*cells[k]) for k in pool]); base = pm[rng.integers(0, len(pm), n_shuffle)].mean()
            same = [nss(*cells[k]) for k in pool if k[2] == b]
            cc = np.corrcoef(maps[(p, c, b)].ravel(), C.ravel())[0, 1]
            out.append((p, c, b, len(cells[(p, c, b)][0]), obs, base, np.mean(same) if same else np.nan, cc))
    R = pd.DataFrame(out, columns=['pid', 'clip', 'bin', 'n', 'nss', 'base', 'base_same', 'cc'])
    R['surplus'] = R.nss - R.base; R['surplus_same'] = R.nss - R.base_same
    return R

def excerpt_level(R, col='surplus'):
    return R.groupby(['pid', 'clip'])[col].mean().unstack().reindex(columns=CLIPS)

RB = build(INC)                                                       # primary
ES = excerpt_level(RB)
print('bins scored:', len(RB), '| participants:', RB.pid.nunique())
display(RB.groupby('clip')[['nss', 'base', 'surplus', 'surplus_same', 'cc']].mean().reindex(CLIPS).round(3))

### 9. H1a, sensitivity versions, adequacy, leave-one-out

**فارسی:** H1a: تفاضل درون‌فردی پراستاندارد منهای کم‌استاندارد، بوت‌استرپ، نسخه‌های حساسیت، شرایط کفایت، حذف یک‌به‌یک.

*Thesis: 3-9-6, 3-9-7; 5-5, 5-6; Tables 5-6 to 5-10*

In [ ]:
# 9. H1a: within-person high minus low (K-pop only), bootstrap CI, sensitivity versions, adequacy conditions, leave-one-out
# Thesis: 3-9-6, 3-9-7; 5-5, 5-6; Tables 5-6 to 5-10
HI = CK.loc[(CK.corpus == 'kpop') & (CK.level == 'high'), 'clip'].tolist(); LO = CK.loc[(CK.corpus == 'kpop') & (CK.level == 'low'), 'clip'].tolist()

def h1a(E, hi=HI, lo=LO, boot=N_BOOT, seed=7):
    d = (E[hi].mean(1) - E[lo].mean(1)).dropna()
    if len(d) == 0: return dict(n=0)
    rng = np.random.default_rng(seed); X = E.loc[d.index]; bs = []
    for _ in range(boot):
        pi = rng.integers(0, len(X), len(X)); h = rng.choice(hi, len(hi)); l = rng.choice(lo, len(lo))
        bs.append((X.iloc[pi][h].mean(1) - X.iloc[pi][l].mean(1)).mean())
    lo_, hi_ = np.nanpercentile(bs, [2.5, 97.5]); m = d.mean()
    verdict = 'consistent with H1a' if m > 0 and lo_ > 0 else ('inconsistent with H1a' if m <= 0 else 'indeterminate')
    return dict(n=len(d), mean_diff=m, ci_low=lo_, ci_high=hi_, n_positive=int((d > 0).sum()), verdict=verdict)

H1A = [dict(version='primary (sigma 4 deg, first second excluded)', **h1a(ES))]
F_hi = CK.loc[CK['clip'].isin(HI) & (CK.female == 1), 'clip'].tolist(); F_lo = CK.loc[CK['clip'].isin(LO) & (CK.female == 1), 'clip'].tolist()
M_hi = [c for c in HI if c not in F_hi]; M_lo = [c for c in LO if c not in F_lo]
H1A += [dict(version='female groups only', **h1a(ES, F_hi, F_lo)), dict(version='male groups only', **h1a(ES, M_hi, M_lo))]
H1A += [dict(version='same-relative-second baseline', **h1a(excerpt_level(RB, 'surplus_same')))]
H1A += [dict(version='first second included', **h1a(excerpt_level(build(INC, bins=range(0, EXC_S)))))]
p3 = P.loc[P.included & (P.acc_deg <= 3), 'pid'].tolist()
H1A += [dict(version='sigma 3 deg, participants with error <= 3 deg', **h1a(excerpt_level(build(p3, sigma_deg=3.0))) if len(p3) >= 3 else dict(n=len(p3)))]
frac = float(np.nanmedian([SIGMA_DEG * PPD[p] / RW[(p, c)] for p, c in TRI[['pid', 'clip']].itertuples(index=False)]))
H1A += [dict(version=f'sigma in video-width units ({frac:.3f})', **h1a(excerpt_level(build(INC, sigma_frac=frac))))]
H1A += [dict(version='CC instead of NSS (no baseline)', **h1a(excerpt_level(RB, 'cc')))]
# sensitivity rows added before data collection (preregistration log)
pr = P.loc[P.included & ~P.geometry_nominal, 'pid'].tolist()
H1A += [dict(version=f'participants with nominal (fallback) geometry excluded ({len(INC) - len(pr)} dropped)', **h1a(excerpt_level(build(pr))) if len(pr) >= 3 else dict(n=len(pr)))]
H1A += [dict(version='at least 8 samples per bin', **h1a(excerpt_level(build(INC, min_samples=8))))]
H1A += [dict(version=f'{POST_CUT} s after each cut dropped', **h1a(excerpt_level(build(INC, drop_post_cut=True))))]
G_nd = G.copy(); G_nd['u'] = (G_nd.x - G_nd.rect_l) / G_nd.rect_w; G_nd['v'] = (G_nd.y - G_nd.rect_t) / G_nd.rect_h
G_nd['in_video'] = G_nd.u.between(0, 1, inclusive='left') & G_nd.v.between(0, 1, inclusive='left')
H1A += [dict(version='drift-uncorrected gaze (primary uses drift-corrected)', **h1a(excerpt_level(build(INC, g_source=G_nd))))]
def fixations(g, disp_deg=1.5, min_dur=0.1):   # I-DT: keep samples inside windows with dispersion (x range + y range) <= disp_deg lasting >= min_dur s
    keep = []
    for _, x in g.sort_values('vt').groupby(['pid', 'clip']):
        t, px, py, lim = x.vt.values, x.xc.values, x.yc.values, disp_deg * x.ppd.iloc[0]; i, n = 0, len(x)
        while i < n:
            j = i
            while j + 1 < n and np.ptp(px[i:j + 2]) + np.ptp(py[i:j + 2]) <= lim: j += 1
            if t[j] - t[i] >= min_dur: keep.extend(x.index[i:j + 1]); i = j + 1
            else: i += 1
    return g.loc[keep].copy()
G_fix = fixations(G)
H1A += [dict(version=f'fixation samples only (I-DT, 1.5 deg, 100 ms; {len(G_fix) / len(G):.0%} of samples kept)', **h1a(excerpt_level(build(INC, g_source=G_fix))))]
for ver in sorted(P.loc[P.included, 'version'].dropna().unique()):   # site version (1.8.4 changed only the timing and feedback of the calibration check; pass rule unchanged)
    pv = P.loc[P.included & (P.version == ver), 'pid'].tolist()
    H1A += [dict(version=f'site version {ver} only ({len(pv)} participants)', **h1a(excerpt_level(build(pv))) if len(pv) >= 3 else dict(n=len(pv)))]
H1A = pd.DataFrame(H1A); display(H1A.round(3))

# adequacy conditions (OSF 5.5)
def split_half(R, a, b):
    x = R[R.bin.isin(a)].groupby(['pid', 'clip']).surplus.mean(); y = R[R.bin.isin(b)].groupby(['pid', 'clip']).surplus.mean()
    j = pd.concat([x, y], axis=1).dropna(); r = j.corr().iloc[0, 1] if len(j) > 2 else np.nan
    return 2 * r / (1 + r) if r == r else np.nan
odd, even = range(1, EXC_S, 2), range(2, EXC_S, 2)
lvl = RB.merge(CK[['clip', 'corpus', 'level']], on='clip')
above = lvl[lvl.corpus == 'kpop'].groupby('level').apply(lambda x: pd.Series(dict(nss=x.nss.mean(), base=x.base.mean(), diff=(x.nss - x.base).mean())))
ADEQ = pd.DataFrame([
    dict(condition='valid participants >= 25', value=len(INC), met=len(INC) >= 25),
    dict(condition='split-half reliability (odd/even bins, Spearman-Brown) >= 0.50', value=split_half(RB, odd, even), met=split_half(RB, odd, even) >= 0.5),
    dict(condition='split-half reliability (first/second half), reported', value=split_half(RB, range(1, 10), range(10, EXC_S)), met=np.nan),
    dict(condition='K-pop level gap >= 1 SD (excerpt index)', value=2.4, met=True),
    dict(condition='observed NSS above baseline, high level', value=above.loc['high', 'diff'] if 'high' in above.index else np.nan, met=bool(above.loc['high', 'diff'] > 0) if 'high' in above.index else False),
    dict(condition='observed NSS above baseline, low level', value=above.loc['low', 'diff'] if 'low' in above.index else np.nan, met=bool(above.loc['low', 'diff'] > 0) if 'low' in above.index else False)])
display(ADEQ.round(3))
print('ALL ADEQUACY CONDITIONS MET' if ADEQ.met.dropna().astype(bool).all() else 'Adequacy not met: report H1a with this label and no judgment')

# leave-one-participant-out and leave-one-excerpt-out (consensus maps rebuilt each time)
LOO = [dict(left_out='participant ' + p, **h1a(excerpt_level(build([q for q in INC if q != p])), boot=200)) for p in INC]
LOO += [dict(left_out='excerpt ' + c, **h1a(excerpt_level(build(INC, clips=[k for k in CLIPS if k != c])), [h for h in HI if h != c], [l for l in LO if l != c], boot=200)) for c in HI + LO]
LOO = pd.DataFrame(LOO); display(LOO[['left_out', 'n', 'mean_diff', 'ci_low', 'ci_high']].round(3))

d_ind = (ES[HI].mean(1) - ES[LO].mean(1)).dropna().sort_values()
plt.figure(figsize=(7, 3.2)); plt.bar(range(len(d_ind)), d_ind.values, color=['#2a9d8f' if v > 0 else '#e76f51' for v in d_ind.values])
plt.axhline(0, color='k', lw=.8); plt.ylabel('surplus high minus low'); plt.xlabel('participants'); plt.title('H1a: within-person differences'); plt.tight_layout()
plt.savefig(FIG / 'H1a_individual_differences.png', dpi=150); plt.show()

### 10. Participant saturation (optional, heavy)

**فارسی:** (اختیاری، سنگین) اشباع شرکت‌کننده‌ها: پایداری اجماع و برآورد H1a در زیرگروه‌ها.

*Thesis: 5-7; Table 5-11*

In [ ]:
# 10. Saturation over participants -> gaze/participant_saturation.xlsx (optional, heavy)
# Thesis: 5-7; Table 5-11
# Would more viewers change the result? Random subsets of k included participants (k = 10, 15, 20, ... , all; 40 draws each).
# For every draw the consensus maps and the surplus are rebuilt from the subset only, then:
#   rho_clips  = Spearman rho of the 12 excerpt-level mean surpluses with the full-sample values (stability of the consensus)
#   mean_diff  = H1a estimate (high minus low) in the subset; half_width = 1.96 * SD(d) / sqrt(k) (precision of H1a)
# Rule fixed in advance: the consensus counts as saturated from the smallest k whose 10th percentile of rho_clips >= .90.
# Heavy: about (number of sizes x 40) rebuilds; lower SAT_REPS to 20 if it is too slow.
from scipy.stats import spearmanr
SAT_REPS, SAT_SHUFFLE = 40, 20
full_clip = ES.mean(0)
Nall = len(INC); sizes = sorted(set([k for k in range(10, Nall, 5)] + [Nall]))
rng_p = np.random.default_rng(2026); rows = []
for k in sizes:
    for r in range(SAT_REPS if k < Nall else 1):
        sub = list(rng_p.choice(INC, k, replace=False)) if k < Nall else list(INC)
        E = excerpt_level(build(sub, n_shuffle=SAT_SHUFFLE, seed=r))
        d = (E[HI].mean(1) - E[LO].mean(1)).dropna()
        rows.append(dict(k=k, rep=r, rho_clips=spearmanr(E.mean(0).reindex(CLIPS), full_clip.reindex(CLIPS), nan_policy='omit')[0],
                         mean_diff=d.mean(), half_width=1.96 * d.std(ddof=1) / np.sqrt(len(d))))
    print('k =', k, 'done')
PSAT_RAW = pd.DataFrame(rows)
PSAT = PSAT_RAW.groupby('k').agg(rho_clips_median=('rho_clips', 'median'), rho_clips_p10=('rho_clips', lambda x: np.percentile(x, 10)),
                                 mean_diff_median=('mean_diff', 'median'), mean_diff_p5=('mean_diff', lambda x: np.percentile(x, 5)),
                                 mean_diff_p95=('mean_diff', lambda x: np.percentile(x, 95)), half_width_median=('half_width', 'median')).reset_index()
ok = PSAT.loc[(PSAT.k < Nall) & (PSAT.rho_clips_p10 >= .90), 'k']
K_SAT = int(ok.min()) if len(ok) else None
print('participants needed for a stable consensus (10th pct rho >= .90):', K_SAT if K_SAT else f'NOT reached below {Nall}')
with pd.ExcelWriter(GZ / 'participant_saturation.xlsx', engine='xlsxwriter') as xw:
    PSAT.round(3).to_excel(xw, sheet_name='summary', index=False); PSAT_RAW.to_excel(xw, sheet_name='draws', index=False)
    pd.DataFrame({'item': ['rule', 'saturation_k', 'n_included', 'draws', 'seed'],
                  'value': ['smallest k with 10th pct of rho_clips >= .90', K_SAT, Nall, SAT_REPS, 2026]}).to_excel(xw, sheet_name='rule', index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(PSAT.k, PSAT.rho_clips_median, marker='o'); ax[0].plot(PSAT.k, PSAT.rho_clips_p10, ls='--', label='10th percentile')
ax[0].axhline(.90, c='gray', lw=1); ax[0].set_xlabel('participants'); ax[0].set_ylabel('rho with full-sample excerpt surplus'); ax[0].legend(); ax[0].set_title('Consensus stability')
ax[1].plot(PSAT.k, PSAT.mean_diff_median, marker='o', label='H1a estimate (median)')
ax[1].fill_between(PSAT.k, PSAT.mean_diff_p5, PSAT.mean_diff_p95, alpha=.2, label='5th-95th pct over subsets')
ax[1].plot(PSAT.k, PSAT.half_width_median, ls=':', c='k', label='CI half-width')
ax[1].axhline(0, c='k', lw=.8); ax[1].set_xlabel('participants'); ax[1].legend(); ax[1].set_title('H1a estimate and precision')
plt.tight_layout(); plt.savefig(FIG / 'participant_saturation.png', dpi=150); plt.show()
display(PSAT.round(3))

### 11. H1b mixed model

**فارسی:** H1b: مدل آمیخته‌ی سطح ثانیه با کنترل‌های ادراکی (فقط کی‌پاپ، ۰٫۵ ثانیه پس از برش حذف).

*Thesis: 3-9-6; 5-8; Table 5-12*

In [ ]:
# 11. H1b: bin-level mixed model with perceptual controls (K-pop only, 0.5 s after each cut dropped)
# Thesis: 3-9-6; 5-8; Table 5-12
def mixed(formula, data, groups_pid=True):
    data = data.dropna(subset=[v for v in re.findall(r'[A-Za-z_]+', formula) if v in data.columns]).copy(); data['one'] = 1
    vc = {'pid': '0 + C(pid)', 'clip': '0 + C(clip)'} if groups_pid else {'clip': '0 + C(clip)'}
    try:
        m = smf.mixedlm(formula, data, groups='one', vc_formula=vc, re_formula='0').fit(reml=True, method='lbfgs')
        if not m.converged and groups_pid:  # preregistered fallback: keep participant effect, drop excerpt component
            print('not converged, refit without excerpt component:', formula)
            m = smf.mixedlm(formula, data, groups='one', vc_formula={'pid': '0 + C(pid)'}, re_formula='0').fit(reml=True, method='lbfgs')
        t = pd.DataFrame(dict(coef=m.params, ci_low=m.conf_int()[0], ci_high=m.conf_int()[1], p=m.pvalues)).drop(index=[i for i in m.params.index if 'Var' in i], errors='ignore')
        return t.assign(n_obs=len(data))
    except Exception as e:
        print('model failed:', formula, '|', e); return pd.DataFrame()

RBc = build(INC, clips=HI + LO, drop_post_cut=True)
B = RBc.merge(FEAT, on=['clip', 'bin']).merge(CK[['clip', 'high', 'female']], on='clip')
for v in ['lum', 'con', 'mot', 'cuts', 'onset']: B[v + '_z'] = (B[v] - B[v].mean()) / B[v].std()
H1B = mixed('surplus ~ high + female + lum_z + con_z + mot_z + cuts_z', B)
H1B_audio = mixed('surplus ~ high + female + lum_z + con_z + mot_z + cuts_z + onset_z', B)
X = B[['high', 'female', 'lum_z', 'con_z', 'mot_z', 'cuts_z']].dropna().assign(const=1)
try: VIF = pd.DataFrame(dict(variable=X.columns, vif=[variance_inflation_factor(X.values, i) for i in range(X.shape[1])])).query('variable != "const"')
except Exception as e: print('VIF could not be computed:', e); VIF = pd.DataFrame(dict(variable=[], vif=[]))
display(H1B.round(3)); display(VIF.round(2)); print('exploratory, with audio onset strength:'); display(H1B_audio.round(3))
if len(H1B) and 'high' in H1B.index:
    r = H1B.loc['high']; vif_hi = bool(len(VIF)) and VIF.set_index('variable').vif.max() > 5
    print('H1b:', 'indeterminate (high collinearity)' if vif_hi and not r.ci_low > 0 else ('consistent with H1b' if r.ci_low > 0 else 'not consistent with H1b'))

### 12. Excerpt-level H1b and H1c

**فارسی:** نسخه‌های سطح قطعه‌ی H1b و H1c (وابستگی زمانی ثانیه‌ها).

*Thesis: 3-9-6; 5-8, 5-9*

In [ ]:
# 12. Excerpt-level versions of H1b and H1c (temporal dependence of 1-s bins)
# Thesis: 3-9-6; 5-8, 5-9
# Bins of one participant x excerpt are serially dependent; averaging them to one value per participant x excerpt removes that dependence.
def mixed_ex(formula, data):   # as mixed(), but reports whether the fallback without the excerpt component was used
    data = data.dropna(subset=[v for v in re.findall(r'[A-Za-z_]+', formula) if v in data.columns]).copy(); data['one'] = 1; fb = False
    try:
        m = smf.mixedlm(formula, data, groups='one', vc_formula={'pid': '0 + C(pid)', 'clip': '0 + C(clip)'}, re_formula='0').fit(reml=True, method='lbfgs')
        if not m.converged:
            fb = True; m = smf.mixedlm(formula, data, groups='one', vc_formula={'pid': '0 + C(pid)'}, re_formula='0').fit(reml=True, method='lbfgs')
        t = pd.DataFrame(dict(coef=m.params, ci_low=m.conf_int()[0], ci_high=m.conf_int()[1], p=m.pvalues)).drop(index=[i for i in m.params.index if 'Var' in i], errors='ignore')
        return t.assign(n_obs=len(data), excerpt_component_dropped=fb, note='precision likely overstated' if fb else '').rename_axis('term').reset_index()
    except Exception as e:
        print('model failed:', formula, '|', e); return pd.DataFrame()
BX = B.groupby(['pid', 'clip']).agg(surplus=('surplus', 'mean'), high=('high', 'first'), female=('female', 'first'), lum=('lum', 'mean'), con=('con', 'mean'), mot=('mot', 'mean'), cuts=('cuts', 'mean')).reset_index()
for v in ['lum', 'con', 'mot', 'cuts']: BX[v + '_z'] = (BX[v] - BX[v].mean()) / BX[v].std()
H1B_EX = mixed_ex('surplus ~ high + female + lum_z + con_z + mot_z + cuts_z', BX).assign(model='H1b excerpt level (8 K-pop)')
CX = RB.groupby(['pid', 'clip']).surplus.mean().reset_index().merge(CK[['clip', 'excerpt_index_z', 'female']], on='clip')
H1C_EX = mixed_ex('surplus ~ excerpt_index_z + female', CX).assign(model='H1c excerpt level (12 excerpts)')
display(H1B_EX.round(4)); display(H1C_EX.round(4))

### 13. H1c, D1, interpretation matrix

**فارسی:** H1c: شیب همگرایی بر شاخص قطعه در ۱۲ قطعه و حساسیت‌ها؛ D1 کی‌پاپ در برابر ایندی؛ ماتریس تفسیر.

*Thesis: 3-9-6, 3-9-7; 5-9, 5-13; Tables 5-13, 5-14, 5-16*

In [ ]:
# 13. H1c: slope of the excerpt-level surplus on the excerpt index (12 excerpts); D1 K-pop vs indie; registered interpretation matrix
# Thesis: 3-9-6, 3-9-7; 5-9, 5-13; Tables 5-13, 5-14, 5-16
EL = ES.stack().rename('surplus').reset_index().merge(CK, on='clip')
FE = FEAT.groupby('clip')[['lum', 'con', 'mot', 'cuts']].mean().reset_index()
EL = EL.merge(FE, on='clip').merge(TRI[['pid', 'clip', 'fam_song', 'fam_group', 'liking']], on=['pid', 'clip'], how='left')
for v in ['excerpt_index_z', 'lum', 'con', 'mot', 'cuts']: EL[v + '_s'] = (EL[v] - EL[v].mean()) / EL[v].std()
H1C = {'primary: 12 excerpts': mixed('surplus ~ excerpt_index_z + female', EL),
       'K-pop only (8)': mixed('surplus ~ excerpt_index_z + female', EL[EL.corpus == 'kpop']),
       'with perceptual features': mixed('surplus ~ excerpt_index_z + female + lum_s + con_s + mot_s + cuts_s', EL),
       'without the dark single-dancer clip': mixed('surplus ~ excerpt_index_z + female', EL[~EL['clip'].isin(C9)]),
       'with familiarity': mixed('surplus ~ excerpt_index_z + female + fam_song + fam_group', EL),
       'exploratory: index + corpus': mixed('surplus ~ excerpt_index_z + kpop + female', EL)}
for c in CLIPS: H1C[f'leave out {c}'] = mixed('surplus ~ excerpt_index_z + female', EL[EL['clip'] != c])
H1C_T = pd.concat({k: v for k, v in H1C.items() if len(v)}, names=['model', 'term']).reset_index()
display(H1C_T[H1C_T.term.isin(['excerpt_index_z', 'kpop'])].round(4))
r = H1C['primary: 12 excerpts']
if len(r): print('H1c:', 'consistent with H1c' if r.loc['excerpt_index_z', 'ci_low'] > 0 else ('inconsistent' if r.loc['excerpt_index_z', 'coef'] <= 0 else 'indeterminate'))

KP = CK.loc[CK.corpus == 'kpop', 'clip'].tolist(); IN = CK.loc[CK.corpus == 'indie', 'clip'].tolist()
D1 = pd.DataFrame([dict(version='all excerpts', **h1a(ES, KP, IN)), dict(version='without the dark single-dancer clip', **h1a(ES, KP, [c for c in IN if c not in C9]))]).rename(columns={'mean_diff': 'kpop_minus_indie'})
D1['verdict'] = 'descriptive only'; display(D1.round(3))
# registered interpretation matrix (ch. 3-9-7): mean surplus of the four groups and the within-person contrasts that place the pattern in a row of the matrix
IH = CK.loc[(CK.corpus == 'indie') & (CK.level.str.contains('high')), 'clip'].tolist(); IL = [c for c in IN if c not in IH]
GRP = {'K-pop high': HI, 'K-pop low': LO, 'indie high-similarity': IH, 'indie low-similarity': IL, 'indie (all)': IN}
MATRIX = pd.DataFrame([dict(group=g, n_clips=len(c), mean_surplus=ES[c].mean(1).mean(), sd_between_persons=ES[c].mean(1).std()) for g, c in GRP.items()])
CONTR = pd.DataFrame([dict(contrast='K-pop high - K-pop low (= H1a)', **h1a(ES, HI, LO)), dict(contrast='K-pop low - indie', **h1a(ES, LO, IN)),
                      dict(contrast='K-pop high - indie', **h1a(ES, HI, IN)), dict(contrast='indie high-sim - indie low-sim', **h1a(ES, IH, IL))])
display(MATRIX.round(3)); display(CONTR.round(3))

m = EL.groupby(['clip', 'corpus', 'level', 'excerpt_index_z']).surplus.agg(['mean', 'sem']).reset_index()
plt.figure(figsize=(6.5, 4))
for (co, lv), x in m.groupby(['corpus', 'level']):
    plt.errorbar(x.excerpt_index_z, x['mean'], yerr=1.96 * x['sem'], fmt='o', capsize=3, label=f'{co} {lv}')
for _, x in m.iterrows(): plt.annotate(x['clip'], (x.excerpt_index_z, x['mean']), fontsize=7, xytext=(3, 3), textcoords='offset points')
plt.xlabel('excerpt index (whole-video z units)'); plt.ylabel('convergence surplus'); plt.axhline(0, color='k', lw=.6); plt.legend(fontsize=8); plt.title('H1c: surplus across the index')
plt.tight_layout(); plt.savefig(FIG / 'H1c_index_vs_surplus.png', dpi=150); plt.show()

### 14. AOI and rival explanation

**فارسی:** ناحیه‌های مورد علاقه: چهره، زیر چهره و دیگر؛ تبیین رقیب.

*Thesis: 5-10; Table 5-15*

In [ ]:
# 14. AOI: face, body (below a detected face) and other; rival explanation (OSF 5.4)
# Thesis: 5-10; Table 5-15
def aoi_of(clip, t, u, v):
    f = FACES[clip]['faces']; fts = FACES[clip]['ftimes']
    if len(f) == 0 or len(fts) == 0: return 'other'
    ft = f[:, 0].astype(float); tt = float(fts[np.argmin(np.abs(fts - t))])
    for _, x, y, w, h, _s in f[np.abs(ft - tt) < 1e-3]:
        if x - .1 * w <= u <= x + 1.1 * w and y - .1 * h <= v <= y + 1.1 * h: return 'face'
    for _, x, y, w, h, _s in f[np.abs(ft - tt) < 1e-3]:
        if x - w <= u <= x + 2 * w and v > y + 1.1 * h: return 'body'
    return 'other'
GA = G[G.in_video & (G.bin >= 1)].copy()
GA['aoi'] = [aoi_of(c, t, u, v) for c, t, u, v in zip(GA['clip'], GA.vt, GA.u, GA.v)]
AOI_P = GA.groupby(['pid', 'clip']).aoi.value_counts(normalize=True).unstack(fill_value=0).reset_index()     # per participant (each counts equally)
AOI = AOI_P.groupby('clip')[[c for c in ['face', 'body', 'other'] if c in AOI_P]].mean().reindex(CLIPS).reset_index().merge(CK[['clip', 'corpus', 'level', 'gender']], on='clip')
has_face = pd.Series({c: float(np.isin(np.round(FACES[c]['ftimes'].astype(float), 3), np.round(FACES[c]['faces'][:, 0].astype(float), 3)).mean()) if len(FACES[c]['ftimes']) else 0.0 for c in CLIPS}, name='share_frames_with_face')
AOI = AOI.merge(has_face.rename_axis('clip').reset_index(), on='clip')
display(AOI.round(3))
print('Rival explanation check (5.4): H1a by group gender is in the H1a table (female / male rows); body share of the female low excerpts:')
display(AOI[(AOI.corpus == 'kpop') & (AOI.gender == 'female')][['clip', 'level', 'face', 'body', 'other']].round(3) if 'body' in AOI else AOI)
AOI.set_index('clip')[[c for c in ['face', 'body', 'other'] if c in AOI]].plot.bar(stacked=True, figsize=(8, 3.2), color=['#f4a261', '#e76f51', '#8ab17d'])
plt.ylabel('share of gaze'); plt.title('AOI dwell share per excerpt'); plt.tight_layout(); plt.savefig(FIG / 'AOI_share.png', dpi=150); plt.show()

### خطای آشکارساز چهره بر حسب جنسیت گروه
خطای آشکارساز چهره (YuNet) به تفکیک جنسیت گروه بدون برچسب‌گذاری دستی فریم‌ها قابل محاسبه نیست. این مورد محاسبه نمی‌شود و در گزارش نهایی به‌عنوان **انحراف از پیش‌ثبت** ذکر می‌شود.

### 15. Static heatmaps

**فارسی:** نقشه‌ی حرارتی ثابت هر قطعه (هموارسازی ۱٫۵ درجه فقط برای تصویر).

*Thesis: 5-11*

In [ ]:
# 15. Static heatmaps: one frame of each excerpt with the pooled gaze density (each participant weighted equally)
# Thesis: 5-11
CMAP = matplotlib.colormaps['turbo']
VIS_SIGMA_DEG = 1.5   # smoothing for the pictures only (the statistics keep the preregistered 4 deg)
def density(clip, Wd, Hd, g, sigma_deg=VIS_SIGMA_DEG):
    m = np.zeros((Hd, Wd))
    for p, x in g.groupby('pid'):
        h = np.zeros((Hd, Wd)); np.add.at(h, (np.minimum((x.v * Hd).astype(int), Hd - 1), np.minimum((x.u * Wd).astype(int), Wd - 1)), 1)
        h = gaussian_filter(h, sigma_deg * PPD[p] / RW[(p, clip)] * Wd, mode='constant'); m += h / h.sum() if h.sum() else 0
    return m
SIG_CELLS = {}
def density_fast(clip, Wd, Hd, g):   # pooled: each participant's samples weighted 1/n, one filter with the median sigma
    if clip not in SIG_CELLS:
        SIG_CELLS[clip] = float(np.median([VIS_SIGMA_DEG * PPD[p] / RW[(p, clip)] for p in g0['pid'].unique() if (p, clip) in RW])) if len(g0 := G[G['clip'] == clip]) else 0.05
    m = np.zeros((Hd, Wd)); w = 1 / g.groupby('pid').pid.transform('size').values
    np.add.at(m, (np.minimum((g.v.values * Hd).astype(int), Hd - 1), np.minimum((g.u.values * Wd).astype(int), Wd - 1)), w)
    return gaussian_filter(m, SIG_CELLS[clip] * Wd, mode='constant')
def overlay(frame, dens, vmax, alpha=0.65):
    d = np.clip(dens / vmax, 0, 1) if vmax > 0 else dens
    col = (CMAP(d)[..., :3][..., ::-1] * 255).astype(np.float32)
    a = (alpha * np.clip((d - 0.15) / 0.85, 0, 1))[..., None]   # low densities transparent
    return (frame.astype(np.float32) * (1 - a) + col * a).astype(np.uint8)
def frame_at(clip, t):
    cap = cv2.VideoCapture(str(CLIPS_DIR / f'{clip}.mp4')); cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000); ok, fr = cap.read(); cap.release(); return fr
fig, ax = plt.subplots(3, 4, figsize=(16, 7.5))
for a, c in zip(ax.ravel(), CLIPS):
    fr = frame_at(c, 10); Hf, Wf = fr.shape[:2]; Wd = 320; Hd = int(Wd * Hf / Wf)
    dens = cv2.resize(density(c, Wd, Hd, GA[GA['clip'] == c]), (Wf, Hf))
    img = overlay(fr, dens, np.percentile(dens, 99.5)); cv2.imwrite(str(FIG / f'heatmap_{c}.jpg'), img)
    k = CK.set_index('clip').loc[c]; a.imshow(img[..., ::-1]); a.set_title(f"{c}  {k.corpus} {k.level}  ({k.excerpt_index_z:+.1f})", fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(FIG / 'heatmaps_all.png', dpi=130); plt.show()

### 16. Heatmap videos (optional)

**فارسی:** (اختیاری) ویدئوی نقشه‌ی حرارتی هر قطعه؛ نام فایل سطح را نشان می‌دهد و منتشر نمی‌شود.

*Thesis: 5-11*

In [ ]:
# 16. Heatmap videos (optional): moving gaze density (+-250 ms) with original sound; file names reveal the level, do not publish
# Thesis: 5-11
WIN = 0.25; RENDER_W = 960
def render(clip):
    src = CLIPS_DIR / f'{clip}.mp4'; cap = cv2.VideoCapture(str(src)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W0, H0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); W = RENDER_W; H = int(round(W * H0 / W0 / 2) * 2)
    g = G[(G['clip'] == clip) & G.in_video]; Wd, Hd = 192, max(8, int(192 * H / W))
    tmp = VID / f'_{clip}.mp4'; vw = cv2.VideoWriter(str(tmp), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    dens = []
    for k in range(n):                                   # pass 1: densities only (small grids)
        t = k / fps; x = g[(g.vt >= t - WIN) & (g.vt < t + WIN)]
        dens.append(density_fast(clip, Wd, Hd, x) if len(x) else np.zeros((Hd, Wd)))
    vmax = np.percentile([d.max() for d in dens if d.max() > 0] or [1], 90)
    cap = cv2.VideoCapture(str(src)); k = 0
    while True:                                          # pass 2: draw
        ok, fr = cap.read()
        if not ok: break
        d = dens[min(k, len(dens) - 1)]; k += 1
        vw.write(overlay(cv2.resize(fr, (W, H)), cv2.resize(d, (W, H)), vmax))
    cap.release(); vw.release()
    k_ = CK.set_index('clip').loc[clip]; out = VID / f"{clip}_{k_.corpus}_{k_.level}_heatmap.mp4"
    subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-i', str(tmp), '-i', str(src), '-map', '0:v', '-map', '1:a?', '-c:v', 'libx264', '-crf', '20', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-shortest', str(out)], check=True)
    tmp.unlink(); return out
for c in CLIPS: print('written', render(c))

### 17. Exploratory analyses

**فارسی:** تحلیل‌های اکتشافی: برش و ضرب، پنجره‌به‌پنجره، خوشایندی و هواداری، همگرایی معنایی (RQ6).

*Thesis: 5-12*

In [ ]:
# 17. Exploratory analyses (labelled as such): cuts/beats, window-by-window, liking and fans, RQ6 semantic convergence
# Thesis: 5-12
EXP = {}
# (a) gaze dispersion over time (0.2 s steps), locked to cuts and beats: lower dispersion = more convergent
def dispersion(clip, step=0.2):
    g = G[(G['clip'] == clip) & G.in_video]; ts = np.arange(0, EXC_S, step); out = []
    for t in ts:
        x = g[(g.vt >= t) & (g.vt < t + step)].groupby('pid')[['u', 'v']].mean()
        out.append(np.hypot(x.u - x.u.mean(), x.v - x.v.mean()).mean() if len(x) >= 3 else np.nan)
    return pd.Series(out, index=np.round(ts, 1))
DISP = {c: dispersion(c) for c in CLIPS}
def locked(events):
    rows = []
    for c in CLIPS:
        s = DISP[c]
        for e in events[c]:
            for lag in np.round(np.arange(-1, 2.01, 0.2), 1):
                t = round(round((e + lag) / 0.2) * 0.2, 1)
                if t in s.index: rows.append((lag, s[t] - s.mean()))
    return pd.DataFrame(rows, columns=['lag_s', 'dispersion_rel']).groupby('lag_s').dispersion_rel.agg(['mean', 'count'])
EXP['locked_to_cuts'] = locked(CUTS).reset_index(); EXP['locked_to_beats'] = locked(BEATS).reset_index()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, (n, d) in zip(ax, [('cuts', EXP['locked_to_cuts']), ('beats', EXP['locked_to_beats'])]):
    a.plot(d.lag_s, d['mean'], marker='o'); a.axvline(0, color='k', lw=.6); a.set_title(f'gaze dispersion around {n}'); a.set_xlabel('s from event')
plt.tight_layout(); plt.savefig(FIG / 'dispersion_cuts_beats.png', dpi=150); plt.show()

# (b) window-by-window: index of each 4-s model window inside the excerpt vs surplus of the bins it covers
try:
    def l2n(x): return x / np.linalg.norm(x, axis=-1, keepdims=True)
    meta = pd.read_excel(METADATA, sheet_name='Metadata_105'); kdir = OUT / 'per_video' / 'kpop'; idir = OUT / 'per_video' / 'kindie'
    def load(d):
        out = {}
        for v in d.iterdir():
            if (v / 'info.json').exists() and (v / 'embeddings.npz').exists():
                fn = json.loads((v / 'info.json').read_text(encoding='utf-8'))['file_name']; e = np.load(v / 'embeddings.npz', allow_pickle=True)
                out[fn] = (l2n(e['clip_embeddings'].astype(np.float64)), e['clip_starts'], e['clip_ends'])
        return out
    KE, IE = load(kdir), load(idir); names = list(meta.file_name)
    V = np.stack([l2n(KE[n][0].mean(0)) for n in names]); fem = (meta.gender.str.lower() == 'female').to_numpy()
    S_ = V @ V.T; np.fill_diagonal(S_, np.nan); vid_idx = 0.5 * np.nanmean(S_[:, fem], 1) + 0.5 * np.nanmean(S_[:, ~fem], 1); mu, sd = vid_idx.mean(), vid_idx.std()
    rows = []
    for _, k in CK.iterrows():
        W_, st, en = (KE if k.corpus == 'kpop' else IE)[k.file_name]; c0 = k.chorus_start
        for w, s0, e0 in zip(W_, st, en):
            if s0 >= c0 and e0 <= c0 + EXC_S:
                s = w @ V.T
                if k.corpus == 'kpop': s[names.index(k.file_name)] = np.nan
                wi = (0.5 * np.nanmean(s[fem]) + 0.5 * np.nanmean(s[~fem]) - mu) / sd
                b0, b1 = int(s0 - c0), int(e0 - c0)
                rows.append(dict(clip=k['clip'], window_start=s0 - c0, window_index_z=wi, surplus=RB[(RB['clip'] == k['clip']) & RB.bin.between(b0, b1 - 1)].surplus.mean()))
    WW = pd.DataFrame(rows); EXP['window_by_window'] = WW
    r = spearmanr(WW.window_index_z, WW.surplus, nan_policy='omit'); print(f'window-by-window: rho = {r[0]:+.2f} (n windows = {len(WW)})')
except Exception as e:
    print('window-by-window analysis skipped:', e)

# (c) liking ~ level and familiarity (processing fluency); fan vs non-fan
LK = TRI.merge(CK, on='clip').merge(P[['pid', 'fan', 'kpop_freq']], on='pid')
EXP['liking_model'] = mixed('liking ~ excerpt_index_z + kpop + fam_song + fam_group', LK).reset_index()
EL2 = EL.merge(P[['pid', 'fan']], on='pid'); EL2['fan_group'] = np.where(EL2.fan >= 4, 'fan (4-5)', 'non-fan (1-3)')
EXP['fan_descriptives'] = EL2.groupby(['fan_group', 'corpus', 'level']).surplus.agg(['mean', 'count']).reset_index()
display(EXP['liking_model'].round(3)); display(EXP['fan_descriptives'].round(3))

# (d) RQ6: semantic convergence of the end-of-session sentences (multilingual sentence embeddings, leave-one-out)
try:
    try: from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True); from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    SN = TRI[TRI.sentence.str.strip().str.len() > 0][['pid', 'clip', 'sentence']].reset_index(drop=True)
    E_ = st_model.encode(SN.sentence.tolist(), normalize_embeddings=True); SN['sem_conv'] = np.nan
    for c, idx in SN.groupby('clip').groups.items():
        idx = list(idx)
        if len(idx) < 3: continue
        M = E_[idx] @ E_[idx].T; np.fill_diagonal(M, np.nan); SN.loc[idx, 'sem_conv'] = np.nanmean(M, 1)
    SC = SN.merge(ES.stack().rename('surplus').reset_index(), on=['pid', 'clip'])
    per_clip = SC.groupby('clip')[['sem_conv', 'surplus']].mean()
    r1 = spearmanr(per_clip.sem_conv, per_clip.surplus, nan_policy='omit'); r2 = spearmanr(SC.sem_conv, SC.surplus, nan_policy='omit')
    EXP['RQ6_semantic'] = per_clip.reset_index().merge(CK[['clip', 'corpus', 'level']], on='clip')
    print(f'RQ6: semantic vs gaze convergence, across excerpts rho = {r1[0]:+.2f}; across participant x excerpt rho = {r2[0]:+.2f} (n sentences = {len(SN)})')
    display(EXP['RQ6_semantic'].round(3))
except Exception as e:
    print('RQ6 skipped:', e)

### 18. Save all tables

**فارسی:** ذخیره‌ی همه‌ی جدول‌ها در gaze_results.xlsx.

*Thesis: chapter 5*

In [ ]:
# 18. Save every table -> gaze/gaze_results.xlsx
# Thesis: chapter 5
with pd.ExcelWriter(GZ / 'gaze_results.xlsx', engine='xlsxwriter') as xw:
    flow.to_excel(xw, sheet_name='data_flow', index=False)
    P.to_excel(xw, sheet_name='participants', index=False)
    Q.to_excel(xw, sheet_name='data_quality', index=False)
    TR.drop(columns=['frame_log', 'cross']).to_excel(xw, sheet_name='trials', index=False)
    CB.to_excel(xw, sheet_name='central_bias', index=False)
    RB.to_excel(xw, sheet_name='surplus_bins', index=False)
    ES.reset_index().to_excel(xw, sheet_name='surplus_excerpt', index=False)
    H1A.to_excel(xw, sheet_name='H1a', index=False); ADEQ.to_excel(xw, sheet_name='adequacy', index=False); LOO.to_excel(xw, sheet_name='H1a_leave_one_out', index=False)
    H1B.reset_index().to_excel(xw, sheet_name='H1b', index=False); VIF.to_excel(xw, sheet_name='H1b_VIF', index=False); H1B_audio.reset_index().to_excel(xw, sheet_name='H1b_audio_explor', index=False)
    H1B_EX.to_excel(xw, sheet_name='H1b_excerpt_level', index=False); H1C_EX.to_excel(xw, sheet_name='H1c_excerpt_level', index=False)
    H1C_T.to_excel(xw, sheet_name='H1c', index=False); D1.to_excel(xw, sheet_name='D1_descriptive', index=False)
    MATRIX.to_excel(xw, sheet_name='matrix_groups', index=False); CONTR.to_excel(xw, sheet_name='matrix_contrasts', index=False)
    AOI.to_excel(xw, sheet_name='AOI', index=False); AOI_P.to_excel(xw, sheet_name='AOI_participant', index=False)
    FEAT.to_excel(xw, sheet_name='clip_features', index=False)
    for k, v in EXP.items(): v.to_excel(xw, sheet_name=('X_' + k)[:31], index=False)
print('saved:', GZ / 'gaze_results.xlsx', '| figures:', FIG, '| heatmap videos:', VID)

### 19. Additional sensitivity versions, H1c 12-point permutation, H1b with participant × excerpt

**فارسی:** نسخه‌های حساسیت تکمیلی H1a، آزمون جایگشت H1c با ۱۲ نقطه، H1b با جزء تصادفی فرد × قطعه، رشد خطای اعتبارسنجی.

*Thesis: 5-6, 5-8, 5-9; Tables 5-9, 5-13*

In [ ]:
# 19. Additional analyses: screened-out participant, three-validation exclusion, per-person smoothing, gender-stratified bootstrap,
#   viewer gender (exploratory), H1c over 12 points with permutation p, H1b with participant x excerpt component, validation-error growth
#   -> gaze/gaze_additional_results.xlsx, gaze/pip_freeze_gaze.txt
# Thesis: 5-6, 5-8, 5-9; Tables 5-9, 5-13
import subprocess, time
t0 = time.time(); EXTRA = {}

def make_G(pids, trials):   # same steps as step 7 (drift-corrected gaze in video coordinates)
    g = S[S.pid.isin(pids)].merge(trials[['pid', 'clip', 'drift_dx', 'drift_dy', 'rect_l', 'rect_t', 'rect_w', 'rect_h', 'vw', 'vh']], on=['pid', 'clip'])
    g['xc'] = g.x - g.drift_dx; g['yc'] = g.y - g.drift_dy
    g['u'] = (g.xc - g.rect_l) / g.rect_w; g['v'] = (g.yc - g.rect_t) / g.rect_h
    g['in_video'] = g.u.between(0, 1, inclusive='left') & g.v.between(0, 1, inclusive='left')
    g = g[(g.vt >= 0) & (g.vt < EXC_S)].copy(); g['bin'] = np.floor(g.vt).astype(int)
    g['post_cut'] = [any(0 <= vt - c < POST_CUT for c in CUTS[cl]) for vt, cl in zip(g.vt, g['clip'])]
    return g.merge(P[['pid', 'ppd']], on='pid')

ROWS = []
def add(label, res): ROWS.append(dict(version=label, **res)); print(f'{time.time() - t0:6.0f}s  {label}')

# (2) the participant screened out by the site although all 12 excerpts were watched
XP = P.loc[P.excl.str.startswith('screened out') & (P.n_trials == len(CLIPS)) & (P.acc_deg <= ACC_MAX_DEG), 'pid'].tolist()
print('screened-out participants with all excerpts and accuracy <= 4 deg:', XP)
if XP:
    TRX = TR[TR.pid.isin(XP)].copy(); TRX['trial_ok'] = (TRX.valid_share >= TRIAL_VALID) & ~TRX.left_fullscreen & ~TRX.tab_hidden & (TRX.stalls == 0)
    TRX = TRX[TRX.trial_ok]
    for r in TRX.itertuples(): RW[(r.pid, r.clip)] = r.rect_w
    G_x = pd.concat([G, make_G(XP, TRX)], ignore_index=True)
    ES_x = excerpt_level(build(INC + XP, g_source=G_x))
    add(f'with the screened-out participant(s) {XP}', h1a(ES_x))
    EXTRA['ES_with_excluded'] = ES_x.reset_index()

# (3) accuracy from all three validations (start, mid-session, end)
P['err3_deg'] = P[['acc_deg', 'mid_err_deg', 'end_err_deg']].mean(axis=1, skipna=True)
p3v = P.loc[P.included & (P.err3_deg <= ACC_MAX_DEG), 'pid'].tolist()
add(f'exclusion by mean of three validations <= 4 deg ({len(p3v)} participants)', h1a(excerpt_level(build(p3v))) if len(p3v) >= 3 else dict(n=len(p3v)))
sig = {p: max(SIGMA_DEG, e) for p, e in P.set_index('pid').err3_deg.items() if e == e}
add('per-person smoothing: sigma = max(4, mean of three validation errors) deg', h1a(excerpt_level(build(INC, sigma_by_pid=sig))))

# (4) bootstrap stratified by group gender: each draw keeps one female and one male pair (2 + 2 excerpts per level)
def h1a_strat(E, boot=N_BOOT, seed=7):
    d = (E[HI].mean(1) - E[LO].mean(1)).dropna(); X = E.loc[d.index]; rng = np.random.default_rng(seed); bs = []
    for _ in range(boot):
        pi = rng.integers(0, len(X), len(X))
        h = list(rng.choice(F_hi, len(F_hi))) + list(rng.choice(M_hi, len(M_hi))); l = list(rng.choice(F_lo, len(F_lo))) + list(rng.choice(M_lo, len(M_lo)))
        bs.append((X.iloc[pi][h].mean(1) - X.iloc[pi][l].mean(1)).mean())
    lo_, hi_ = np.nanpercentile(bs, [2.5, 97.5]); m = d.mean()
    return dict(n=len(d), mean_diff=m, ci_low=lo_, ci_high=hi_, n_positive=int((d > 0).sum()),
                verdict='consistent with H1a' if m > 0 and lo_ > 0 else ('inconsistent with H1a' if m <= 0 else 'indeterminate'))
def h1a_people_only(E, boot=N_BOOT, seed=7):
    d = (E[HI].mean(1) - E[LO].mean(1)).dropna(); rng = np.random.default_rng(seed)
    bs = [d.iloc[rng.integers(0, len(d), len(d))].mean() for _ in range(boot)]; lo_, hi_ = np.percentile(bs, [2.5, 97.5])
    return dict(n=len(d), mean_diff=d.mean(), ci_low=lo_, ci_high=hi_, n_positive=int((d > 0).sum()), verdict='reference only (excerpts fixed)')
add('bootstrap stratified by group gender', h1a_strat(ES))
add('bootstrap over participants only (excerpts treated as fixed; reference)', h1a_people_only(ES))

# (5) viewer gender (exploratory)
for gv, x in P[P.included].groupby('gender'):
    ids = [p for p in x.pid if p in ES.index]
    if len(ids) >= 3: add(f'viewer gender = {gv} ({len(ids)} viewers; exploratory)', h1a(ES.loc[ids]))
EXTRA['H1a_additional'] = pd.DataFrame(ROWS)

# (6) H1c with the 12 excerpts as the units: slope of the excerpt mean surplus on the index, permutation p and participant bootstrap CI
xi = CK.set_index('clip').excerpt_index_z.reindex(CLIPS).to_numpy(); ym = ES[CLIPS].mean().to_numpy()
slope = lambda x, y: np.polyfit(x, y, 1)[0]
b0 = slope(xi, ym); rng = np.random.default_rng(11)
perm = np.array([slope(rng.permutation(xi), ym) for _ in range(20000)])
bs = np.array([slope(xi, ES.iloc[rng.integers(0, len(ES), len(ES))][CLIPS].mean().to_numpy()) for _ in range(N_BOOT)])
bs_both = []
for _ in range(N_BOOT):   # participants and excerpts both resampled
    k = rng.integers(0, 12, 12); E_ = ES.iloc[rng.integers(0, len(ES), len(ES))][CLIPS].mean().to_numpy()
    if len(set(xi[k])) > 2: bs_both.append(slope(xi[k], E_[k]))
rs, rp = spearmanr(xi, ym)
EXTRA['H1c_12_points'] = pd.DataFrame([
    dict(quantity='slope of excerpt mean surplus on excerpt index (12 points)', value=b0),
    dict(quantity='permutation p, two-sided (index shuffled over the 12 excerpts, 20000 draws)', value=np.mean(np.abs(perm) >= abs(b0))),
    dict(quantity='permutation p, one-sided (slope > 0)', value=np.mean(perm >= b0)),
    dict(quantity='95% CI, participants resampled', value=f'{np.percentile(bs, 2.5):.4f} to {np.percentile(bs, 97.5):.4f}'),
    dict(quantity='95% CI, participants and excerpts resampled', value=f'{np.percentile(bs_both, 2.5):.4f} to {np.percentile(bs_both, 97.5):.4f}'),
    dict(quantity='Spearman rho (12 excerpts)', value=rs), dict(quantity='Spearman p', value=rp)])
print('H1c 12 points done', f'{time.time() - t0:.0f}s')

# (7) H1b with a participant x excerpt random component (bins of one viewer on one excerpt are not independent)
Bp = B.copy(); Bp['pc'] = Bp.pid.astype(str) + '_' + Bp['clip'].astype(str); Bp['one'] = 1
f_ = 'surplus ~ high + female + lum_z + con_z + mot_z + cuts_z'
Bp = Bp.dropna(subset=['surplus', 'lum_z', 'con_z', 'mot_z', 'cuts_z'])
try:
    m = smf.mixedlm(f_, Bp, groups='one', vc_formula={'pid': '0 + C(pid)', 'clip': '0 + C(clip)', 'pc': '0 + C(pc)'}, re_formula='0').fit(reml=True, method='lbfgs')
    EXTRA['H1b_pid_x_excerpt'] = pd.DataFrame(dict(coef=m.params, ci_low=m.conf_int()[0], ci_high=m.conf_int()[1], p=m.pvalues)).rename_axis('term').reset_index().assign(converged=m.converged, n_obs=len(Bp))
except Exception as e:
    print('H1b with pid x excerpt failed:', e); EXTRA['H1b_pid_x_excerpt'] = pd.DataFrame([dict(term='failed', note=str(e))])
print('H1b done', f'{time.time() - t0:.0f}s')

# (8) why did the validation error grow during the session?
Qx = P[P.included].copy()
Qx['growth_mid'] = Qx.mid_err_deg - Qx.acc_deg; Qx['growth_end'] = Qx.end_err_deg - Qx.acc_deg
Qx = Qx.merge(Q[['pid', 'drift_deg', 'rate_hz', 'frame_lag_ms']], on='pid', how='left')
cor = []
for y in ['growth_mid', 'growth_end', 'end_err_deg']:
    for x in ['acc_deg', 'prec_deg', 'n_calibrations', 'view_dist_mm', 'ppd', 'rate_hz', 'drift_deg', 'preload_s', 'geometry_nominal', 'no_card']:
        j = Qx[[x, y]].dropna().astype(float)
        if len(j) > 5 and j[x].std() > 0: r, p = spearmanr(j[x], j[y]); cor.append(dict(outcome=y, predictor=x, n=len(j), rho=r, p=p))
EXTRA['error_growth_correlates'] = pd.DataFrame(cor)
Tq = TRI.merge(P[['pid', 'ppd']], on='pid'); Tq['drift_deg'] = np.hypot(Tq.drift_dx, Tq.drift_dy) / Tq.ppd
EXTRA['drift_by_trial_order'] = Tq.groupby('order').agg(n=('pid', 'size'), drift_deg_mean=('drift_deg', 'mean'), drift_deg_median=('drift_deg', 'median')).reset_index()
try:
    md = smf.mixedlm('drift_deg ~ order', Tq.dropna(subset=['drift_deg']), groups='pid').fit()
    EXTRA['drift_order_model'] = pd.DataFrame(dict(coef=md.params, ci_low=md.conf_int()[0], ci_high=md.conf_int()[1], p=md.pvalues)).rename_axis('term').reset_index()
except Exception as e: print('drift model failed:', e)
EXTRA['participant_errors'] = Qx[['pid', 'acc_deg', 'mid_err_deg', 'end_err_deg', 'err3_deg', 'growth_mid', 'growth_end', 'n_calibrations', 'drift_deg']]

# (9) package versions for Appendix C
(GZ / 'pip_freeze_gaze.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)

with pd.ExcelWriter(GZ / 'gaze_additional_results.xlsx', engine='xlsxwriter') as xw:
    for k, v in EXTRA.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
display(EXTRA['H1a_additional'].round(3)); display(EXTRA['H1c_12_points']); display(EXTRA['H1b_pid_x_excerpt'].round(4))
print('saved:', GZ / 'gaze_additional_results.xlsx', '| total', f'{(time.time() - t0) / 60:.1f} min')

### 20. Further checks: time-shifted baseline, spatial concentration, timing, familiarity, order

**فارسی:** وارسی‌های بیشتر: خط پایه‌ی جابه‌جاشده در زمان، تمرکز فضایی اجماع، بذرها، حذف یک‌به‌یک ۲۰۰۰ تکرار، هندسه، جابه‌جایی، آشنایی، ترتیب، تأخیر، پایایی درون قطعه، اندازه‌ی چهره.

*Thesis: 5-6-2, 5-9, 5-10*

In [ ]:
# 20. Further checks of the gaze analysis -> gaze/checks_results.xlsx
#   time-shifted baseline (same excerpt, |shift| >= 3 s), consensus concentration (entropy), bootstrap seeds, leave-one-out with 2000 draws,
#   12-point leave-one-out, geometry groups, drift, familiarity, trial order, +100/+200 ms timing, within-excerpt reliability, face size
# Thesis: 5-6-2, 5-9, 5-10
import time, json
t0 = time.time(); CHK = {}
say = lambda m: print(f'{time.time() - t0:6.0f}s  {m}')

# (1) Time-shifted baseline inside the same excerpt: a viewer's gaze at second b scored against the consensus of the
#     other viewers at a different second b' (|b - b'| >= 3) of the SAME excerpt. Surplus then measures moment-to-moment
#     synchrony, not a fixed favourite spot of the frame.
def build_shift(pids, clips=None, min_gap=3, seed=1):
    clips = clips or CLIPS
    g = G[G.pid.isin(pids) & G['clip'].isin(clips) & G.in_video & G.bin.isin(list(range(1, EXC_S)))]
    cells = {k: (x.u.values, x.v.values) for k, x in g.groupby(['pid', 'clip', 'bin']) if len(x) >= MIN_SAMPLES}
    maps = {}
    for (p, c, b), (u, v) in cells.items():
        gh = grid_h(c); m = np.zeros((gh, GRID_W))
        np.add.at(m, (np.minimum((v * gh).astype(int), gh - 1), np.minimum((u * GRID_W).astype(int), GRID_W - 1)), 1)
        m = gaussian_filter(m, SIGMA_DEG * PPD[p] / RW[(p, c)] * GRID_W, mode='constant'); maps[(p, c, b)] = m / m.sum()
    by_cb = {}
    for (p, c, b) in maps: by_cb.setdefault((c, b), []).append(p)
    Z = {}
    for (c, b), ps in by_cb.items():
        if len(ps) < 3: continue
        tot = sum(maps[(q, c, b)] for q in ps)
        for p in ps:
            C = (tot - maps[(p, c, b)]) / (len(ps) - 1); sd = C.std()
            if sd > 0: Z[(p, c, b)] = (C - C.mean()) / sd
    def nss(Zm, c, u, v): gh = grid_h(c); return Zm[np.minimum((v * gh).astype(int), gh - 1), np.minimum((u * GRID_W).astype(int), GRID_W - 1)].mean()
    out = []
    for (p, c, b), Zm in Z.items():
        obs = nss(Zm, c, *cells[(p, c, b)])
        other = [nss(Z[(p, c, b2)], c, *cells[(p, c, b)]) for b2 in range(1, EXC_S) if abs(b2 - b) >= min_gap and (p, c, b2) in Z]
        if other: out.append((p, c, b, obs, np.mean(other)))
    R = pd.DataFrame(out, columns=['pid', 'clip', 'bin', 'nss', 'base_shift']); R['surplus'] = R.nss - R.base_shift
    return R
RS = build_shift(INC); ES_s = excerpt_level(RS)
rows = [dict(analysis='H1a, time-shifted baseline (same excerpt, |shift| >= 3 s)', **h1a(ES_s))]
ELs = ES_s.stack().rename('surplus').reset_index().merge(CK, on='clip')
m_ = mixed('surplus ~ excerpt_index_z + female', ELs)
if len(m_): rows.append(dict(analysis='H1c slope, time-shifted baseline', mean_diff=m_.loc['excerpt_index_z', 'coef'], ci_low=m_.loc['excerpt_index_z', 'ci_low'], ci_high=m_.loc['excerpt_index_z', 'ci_high']))
say('time-shifted baseline done')

# (2) Concentration of the consensus map (entropy; lower = gaze piled on one spot) per excerpt, and H1c controlling for it
def ent(m): q = m.ravel() / m.sum(); q = q[q > 0]; return float(-(q * np.log(q)).sum())
CONC = []
for c in CLIPS:
    g = G[(G['clip'] == c) & G.in_video & G.pid.isin(INC)]; gh = grid_h(c); es = []
    for b, x in g.groupby('bin'):
        m = np.zeros((gh, GRID_W)); np.add.at(m, (np.minimum((x.v.values * gh).astype(int), gh - 1), np.minimum((x.u.values * GRID_W).astype(int), GRID_W - 1)), 1)
        es.append(ent(gaussian_filter(m, 2, mode='constant')))
    CONC.append(dict(clip=c, consensus_entropy=np.mean(es)))
CONC = pd.DataFrame(CONC).merge(CK[['clip', 'excerpt_index_z']], on='clip')
CONC['mean_surplus'] = ES[CLIPS].mean().reindex(CONC['clip']).values
ELc = EL.merge(CONC[['clip', 'consensus_entropy']], on='clip'); ELc['ent_s'] = (ELc.consensus_entropy - ELc.consensus_entropy.mean()) / ELc.consensus_entropy.std()
m_ = mixed('surplus ~ excerpt_index_z + female + ent_s', ELc)
if len(m_): rows.append(dict(analysis='H1c slope controlling consensus concentration', mean_diff=m_.loc['excerpt_index_z', 'coef'], ci_low=m_.loc['excerpt_index_z', 'ci_low'], ci_high=m_.loc['excerpt_index_z', 'ci_high']))
r_, p_ = spearmanr(CONC.consensus_entropy, CONC.mean_surplus); rows.append(dict(analysis='rho(consensus entropy, excerpt mean surplus), 12 excerpts', mean_diff=r_, ci_low=np.nan, ci_high=p_))
CHK['concentration_by_excerpt'] = CONC

# (3) Bootstrap seeds and leave-one-out with 2000 draws
for sd in [1, 2, 3, 4, 5]: rows.append(dict(analysis=f'H1a primary, bootstrap seed {sd}', **h1a(ES, seed=sd)))
LOO2 = [dict(left_out='participant ' + p, **h1a(excerpt_level(build([q for q in INC if q != p])), boot=N_BOOT)) for p in INC]
LOO2 += [dict(left_out='excerpt ' + c, **h1a(excerpt_level(build(INC, clips=[k for k in CLIPS if k != c])), [h for h in HI if h != c], [l for l in LO if l != c], boot=N_BOOT)) for c in HI + LO]
CHK['H1a_leave_one_out_2000'] = pd.DataFrame(LOO2); say('leave-one-out done')

# (4) H1c with 12 points, leaving out one excerpt at a time (permutation p each time)
xi_all = CK.set_index('clip').excerpt_index_z
L12 = []
for c in CLIPS:
    keep = [k for k in CLIPS if k != c]; x_ = xi_all.reindex(keep).to_numpy(); y_ = ES[keep].mean().to_numpy(); b_ = np.polyfit(x_, y_, 1)[0]
    rng = np.random.default_rng(3); pm = np.array([np.polyfit(rng.permutation(x_), y_, 1)[0] for _ in range(10000)])
    L12.append(dict(left_out=c, slope=b_, perm_p_two_sided=np.mean(np.abs(pm) >= abs(b_))))
CHK['H1c_12pt_leave_one_out'] = pd.DataFrame(L12)

# (5) H1a by geometry group (nominal fallback geometry or measured)
for lab, grp in [('measured geometry', P.loc[P.included & ~P.geometry_nominal, 'pid']), ('nominal geometry', P.loc[P.included & P.geometry_nominal, 'pid'])]:
    ids = [p for p in grp if p in ES.index]
    if len(ids) >= 3: rows.append(dict(analysis=f'H1a, {lab} ({len(ids)} viewers; subgroup of the primary estimate)', **h1a(ES.loc[ids])))

# (6) Surplus and drift on the fixation cross: does a larger drift go with a smaller surplus?
Td = TRI.merge(P[['pid', 'ppd']], on='pid'); Td['drift_deg'] = np.hypot(Td.drift_dx, Td.drift_dy) / Td.ppd
Td = Td.merge(ES.stack().rename('surplus').reset_index(), on=['pid', 'clip'])
r_, p_ = spearmanr(Td.drift_deg, Td.surplus); rows.append(dict(analysis='rho(drift on cross, surplus), viewer x excerpt', mean_diff=r_, ci_low=np.nan, ci_high=p_))
pp = Td.groupby('pid')[['drift_deg', 'surplus']].mean(); r_, p_ = spearmanr(pp.drift_deg, pp.surplus)
rows.append(dict(analysis='rho(mean drift, mean surplus), 28 viewers', mean_diff=r_, ci_low=np.nan, ci_high=p_))

# (7) Familiarity: H1a adjusted for the difference in song familiarity between high and low excerpts
fam = TRI.pivot_table(index='pid', columns='clip', values='fam_song')
dd = pd.DataFrame(dict(d=ES[HI].mean(1) - ES[LO].mean(1), fam_d=fam.reindex(columns=HI).mean(1) - fam.reindex(columns=LO).mean(1))).dropna()
dd['fam_c'] = dd.fam_d - dd.fam_d.mean()
mf = smf.ols('d ~ fam_c', dd).fit(); rows.append(dict(analysis='H1a adjusted for familiarity difference (intercept)', mean_diff=mf.params['Intercept'], ci_low=mf.conf_int().loc['Intercept', 0], ci_high=mf.conf_int().loc['Intercept', 1]))
rows.append(dict(analysis='familiarity: mean fam_song high minus low', mean_diff=dd.fam_d.mean(), ci_low=np.nan, ci_high=np.nan))

# (8) Position in the session: H1b and H1c with trial order as a covariate
Bo = B.merge(TRI[['pid', 'clip', 'order']], on=['pid', 'clip']); Bo['order_z'] = (Bo.order - Bo.order.mean()) / Bo.order.std()
m_ = mixed('surplus ~ high + female + lum_z + con_z + mot_z + cuts_z + order_z', Bo)
if len(m_): rows.append(dict(analysis='H1b level coefficient with trial order', mean_diff=m_.loc['high', 'coef'], ci_low=m_.loc['high', 'ci_low'], ci_high=m_.loc['high', 'ci_high']))
Eo = EL.merge(TRI[['pid', 'clip', 'order']], on=['pid', 'clip']); Eo['order_z'] = (Eo.order - Eo.order.mean()) / Eo.order.std()
m_ = mixed('surplus ~ excerpt_index_z + female + order_z', Eo)
if len(m_): rows.append(dict(analysis='H1c slope with trial order', mean_diff=m_.loc['excerpt_index_z', 'coef'], ci_low=m_.loc['excerpt_index_z', 'ci_low'], ci_high=m_.loc['excerpt_index_z', 'ci_high']))

# (9) Timing: gaze shifted by +100 and +200 ms relative to the video (unknown tracker latency)
for lag in [0.1, 0.2]:
    Gl = G.copy(); Gl['vt'] = Gl.vt - lag; Gl = Gl[(Gl.vt >= 0) & (Gl.vt < EXC_S)].copy(); Gl['bin'] = np.floor(Gl.vt).astype(int)
    rows.append(dict(analysis=f'H1a with gaze assigned {int(lag * 1000)} ms earlier in the video', **h1a(excerpt_level(build(INC, g_source=Gl)))))
say('lag versions done')

# (10) Reliability within excerpts: odd/even bins after removing each excerpt's mean
def sb(a, b): j = pd.concat([a, b], axis=1).dropna(); r = j.corr().iloc[0, 1]; return 2 * r / (1 + r)
o = RB[RB.bin % 2 == 1].groupby(['pid', 'clip']).surplus.mean(); e = RB[RB.bin % 2 == 0].groupby(['pid', 'clip']).surplus.mean()
oc = o - o.groupby(level='clip').transform('mean'); ec = e - e.groupby(level='clip').transform('mean')
rows.append(dict(analysis='split-half reliability within excerpts (excerpt means removed)', mean_diff=sb(oc, ec), ci_low=np.nan, ci_high=np.nan))
rows.append(dict(analysis='split-half reliability, raw (as registered)', mean_diff=sb(o, e), ci_low=np.nan, ci_high=np.nan))

# (11) Face box size in degrees (how coarse the face AOI is relative to the tracker error)
fs = []
for c in CLIPS:
    f = FACES[c]['faces']
    if len(f) == 0: continue
    wfrac = float(np.median(f[:, 3].astype(float)) / float(FACES[c]['W']))   # face box width as a share of the video width
    for p, x in TRI[TRI['clip'] == c].groupby('pid'):
        if wfrac == wfrac: fs.append(dict(clip=c, pid=p, face_w_deg=wfrac * x.rect_w.iloc[0] / PPD[p]))
CHK['face_size_deg'] = pd.DataFrame(fs).groupby('clip').face_w_deg.median().reset_index() if fs else pd.DataFrame([dict(note='face width not stored in FACES; skipped')])

# (12) Full-screen refusals and calibration attempts (from the raw records)
fsd = sum(json.dumps(d).count('fs_denied') for d in recs if d['participant_id'] in INC)
rows.append(dict(analysis='count of "fs_denied" entries in included sessions', mean_diff=fsd, ci_low=np.nan, ci_high=np.nan))
cal = []
for d in recs:
    if d['participant_id'] not in INC: continue
    for c in d.get('calibration', []): cal.append(dict(pid=d['participant_id'], **{k: (v if not isinstance(v, (dict, list)) else str(v)[:40]) for k, v in c.items() if k in ('pass', 'label', 'phase', 'when', 'after_trial', 'reason')}))
CHK['calibration_attempts'] = pd.DataFrame(cal) if cal else pd.DataFrame([dict(note='no calibration records')])

CHK['checks'] = pd.DataFrame(rows)
with pd.ExcelWriter(GZ / 'checks_results.xlsx', engine='xlsxwriter') as xw:
    for k, v in CHK.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
display(CHK['checks'].round(3)); print('saved:', GZ / 'checks_results.xlsx', f'| {(time.time() - t0) / 60:.1f} min')